# Repro Paper 1: *ViSoBERT: A Pre-Trained Language Model for Vietnamese Social Media Text Processing* (Nguyen et al., EMNLP 2023)
**Objective:** Strict 1:1 reproduction of the original **ViSoBERT (EMNLP 2023, Table 3 & Table 4)** benchmark experiments on **their exact original Vietnamese social media datasets** (`UIT-VSMEC`, `UIT-VSFC`, `UIT-ViHSD`), using their exact pretrained checkpoints (`uitnlp/visobert` vs. `vinai/phobert-base` & `xlm-roberta-base`) and original hyperparameters.

### Original Paper Reference & Published Target Metrics (EMNLP 2023, Table 4):
- **Authors:** Quoc-Nam Nguyen, Thanh-Cong Phan, Duc-Vu Nguyen, Kiet Van Nguyen (*EMNLP 2023*, pp. 5191–5207, `https://aclanthology.org/2023.emnlp-main.315`)
- **Original Datasets (100% Public on HuggingFace/GitHub):**
  1. **UIT-VSMEC** (Emotion Recognition, 7 classes, $N = 6,927$: `5,548` train / `686` dev / `693` test) — Published Macro-F1: **PhoBERT-base = `65.44%`**, **XLM-R-base = `63.92%`**, **ViSoBERT = `68.87%`** (Weighted-F1: `69.58%`).
  2. **UIT-VSFC** (Sentiment Analysis, 3 classes, $N = 16,175$: `11,426` train / `1,583` dev / `3,166` test) — Published Macro-F1: **PhoBERT-base = `83.90%`**, **XLM-R-base = `82.61%`**, **ViSoBERT = `85.15%`**.
  3. **UIT-ViHSD** (Hate Speech Detection, 3 classes: CLEAN / OFFENSIVE / HATE, $N = 33,400$: `24,048` train / `2,672` dev / `6,680` test) — Published Macro-F1: **PhoBERT-base = `66.07%`**, **XLM-R-base = `65.11%`**, **ViSoBERT = `68.25%`**.
- **Original Hyperparameters (Section 4.2 of EMNLP 2023):**
  - Optimizer: `AdamW`, Learning Rate: `2e-5`, Weight Decay: `0.01`
  - Batch Size: `16` (or `32`), Epochs: `5`, Max Sequence Length: `128` / `256`
  - Loss Function: Standard Cross-Entropy Loss (`nn.CrossEntropyLoss()`)

In [ ]:
# Cell 1: Install exact dependencies on Kaggle (Turn ON Internet & GPU T4/P100)
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

In [ ]:
# Cell 2: Download the Original UIT Social Media Datasets used in ViSoBERT (EMNLP 2023)
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

# Choose which original EMNLP 2023 benchmark task to reproduce:
# Options: "UIT-VSMEC" (fastest, N=6,927), "UIT-VSFC" (N=16,175), "UIT-ViHSD" (N=33,400)
BENCHMARK_TASKS = ["UIT-VSMEC", "UIT-VSFC"]  # Add "UIT-ViHSD" for full 3-task run

PUBLISHED_EMNLP2023_MACRO_F1 = {
    "UIT-VSMEC": {
        "vinai/phobert-base": 65.44,
        "xlm-roberta-base": 63.92,
        "uitnlp/visobert": 68.87,
    },
    "UIT-VSFC": {
        "vinai/phobert-base": 83.90,
        "xlm-roberta-base": 82.61,
        "uitnlp/visobert": 85.15,
    },
    "UIT-ViHSD": {
        "vinai/phobert-base": 66.07,
        "xlm-roberta-base": 65.11,
        "uitnlp/visobert": 68.25,
    },
}

def load_original_uit_dataset(task_name: str):
    if task_name == "UIT-VSMEC":
        # UIT-VSMEC (Emotion Recognition: 7 labels)
        try:
            ds = load_dataset("tridm/UIT-VSMEC")
        except Exception:
            ds = load_dataset("uitnlp/uit-vsmec")
        text_col = "Sentence" if "Sentence" in ds["train"].column_names else "sentence"
        label_col = "Emotion" if "Emotion" in ds["train"].column_names else "label"
    elif task_name == "UIT-VSFC":
        # UIT-VSFC (Vietnamese Students' Feedback Corpus: 3 sentiment labels)
        try:
            ds = load_dataset("uitnlp/uit-vsfc")
        except Exception:
            ds = load_dataset("ura-hcmut/uit-vsfc")
        text_col = "sentence" if "sentence" in ds["train"].column_names else "text"
        label_col = "sentiment" if "sentiment" in ds["train"].column_names else "label"
    elif task_name == "UIT-ViHSD":
        # UIT-ViHSD (Vietnamese Hate Speech Detection: 3 labels)
        ds = load_dataset("sonlam1102/vihsd")
        text_col = "free_text" if "free_text" in ds["train"].column_names else "text"
        label_col = "label_id" if "label_id" in ds["train"].column_names else "label"
    else:
        raise ValueError(f"Unknown task: {task_name}")

    # Convert string labels to integer IDs if needed
    train_df = pd.DataFrame(ds["train"])
    val_key = "validation" if "validation" in ds else ("dev" if "dev" in ds else "test")
    val_df = pd.DataFrame(ds[val_key])
    test_df = pd.DataFrame(ds["test"])

    unique_labels = sorted(train_df[label_col].unique().tolist())
    label2id = {lbl: idx for idx, lbl in enumerate(unique_labels)}

    for df in (train_df, val_df, test_df):
        df["text"] = df[text_col].astype(str)
        df["label"] = df[label_col].map(label2id).astype(int)

    print(f"[{task_name}] Loaded Original Splits -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {len(unique_labels)}")
    return train_df[["text", "label"]], val_df[["text", "label"]], test_df[["text", "label"]], len(unique_labels)

In [ ]:
# Cell 3: Exact Fine-Tuning Pipeline Matching Nguyen et al. (EMNLP 2023, Section 4.2)
from datasets import Dataset

MODELS_TO_COMPARE = [
    "xlm-roberta-base",
    "vinai/phobert-base",
    "uitnlp/visobert",
]

repro_records = []

for task_name in BENCHMARK_TASKS:
    train_df, val_df, test_df, num_labels = load_original_uit_dataset(task_name)
    for model_name in MODELS_TO_COMPARE:
        print("=" * 80)
        print(f"Reproducing {model_name} on Original {task_name} Dataset...")
        set_seed(42)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

        def tokenize_fn(batch):
            return tokenizer(batch["text"], truncation=True, max_length=128)

        train_ds = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
        val_ds = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)
        test_ds = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)
            return {
                "acc": accuracy_score(labels, preds) * 100.0,
                "macro_f1": f1_score(labels, preds, average="macro") * 100.0,
                "weighted_f1": f1_score(labels, preds, average="weighted") * 100.0,
            }

        training_args = TrainingArguments(
            output_dir=f"./ckpt_{task_name}_{model_name.replace('/', '_')}",
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,          # Exact EMNLP 2023 lr
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=5,          # Exact EMNLP 2023 epochs
            weight_decay=0.01,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            fp16=torch.cuda.is_available(),
            report_to="none",
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )

        t0 = time.time()
        trainer.train()
        train_time_s = time.time() - t0

        test_metrics = trainer.evaluate(test_ds)
        kaggle_macro_f1 = test_metrics["eval_macro_f1"]
        kaggle_weighted_f1 = test_metrics["eval_weighted_f1"]
        kaggle_acc = test_metrics["eval_acc"]
        paper_macro_f1 = PUBLISHED_EMNLP2023_MACRO_F1[task_name].get(model_name, np.nan)

        repro_records.append({
            "Dataset (Original)": task_name,
            "Model": model_name,
            "Paper Macro-F1 (%)": paper_macro_f1,
            "Kaggle Macro-F1 (%)": round(kaggle_macro_f1, 2),
            "Delta (Kaggle - Paper)": round(kaggle_macro_f1 - paper_macro_f1, 2),
            "Kaggle Weighted-F1 (%)": round(kaggle_weighted_f1, 2),
            "Kaggle Acc (%)": round(kaggle_acc, 2),
            "Train Time (s)": round(train_time_s, 1),
        })

repro_df = pd.DataFrame(repro_records)
repro_df.to_csv("Repro_Paper1_ViSoBERT_EMNLP2023_Comparison.csv", index=False)
print("\n=== FINAL SIDE-BY-SIDE COMPARISON WITH EMNLP 2023 TABLE 4 ===")
display(repro_df)